# Jour 3 · Construire un agent contrôlé


## Un agent choisit une action

Un pipeline RAG appelle toujours le retriever. Un agent reçoit plusieurs outils et décide s'il doit en appeler un, lequel utiliser et avec quels arguments.

Cette flexibilité ajoute aussi de nouveaux échecs possibles : mauvais outil, mauvais argument, boucle trop longue ou réponse produite sans observation suffisante.


![Pipeline RAG fixe comparé à un agent qui choisit](../assets/jour_01/11_rag_vs_agent.png)

*Utiliser un agent seulement lorsqu'un choix dynamique d'action est réellement nécessaire.*


In [ ]:
import sys
from pathlib import Path


def find_course_root() -> Path:
    """Retrouve la racine, même si Jupyter démarre dans un sous-dossier."""
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "documents").is_dir() and (candidate / "jour_01").is_dir():
            return candidate
    raise FileNotFoundError("Racine du cours introuvable.")


ROOT = find_course_root()
DOCUMENTS_DIR = ROOT / "documents"
OUTPUTS_DIR = ROOT / "outputs"
OUTPUTS_DIR.mkdir(exist_ok=True)

# Les imports Python ne suivent pas automatiquement les chemins de fichiers.
# Cette ligne permet notamment d'importer le package local `apps` depuis un
# notebook lancé dans jour_03 ou corrections/jour_03.
root_text = str(ROOT)
if root_text not in sys.path:
    sys.path.insert(0, root_text)

print("Racine du cours :", ROOT)


In [ ]:
from langchain_ollama import ChatOllama, OllamaEmbeddings

CHAT_MODEL = "qwen2.5:3b"
EMBEDDING_MODEL = "bge-m3"

embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL)
llm = ChatOllama(model=CHAT_MODEL, temperature=0)

print("Modèle de génération :", CHAT_MODEL)
print("Modèle d'embeddings  :", EMBEDDING_MODEL)


In [ ]:
import hashlib

from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


def stable_chunk_id(chunk):
    raw = (
        f"{chunk.metadata.get('source', '')}|"
        f"{chunk.metadata.get('page', '')}|"
        f"{chunk.page_content}"
    )
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()


def load_helpdesk_chunks():
    loaded = []
    for path in sorted(DOCUMENTS_DIR.glob("*.md")):
        if path.name == "README.md":
            continue
        docs = TextLoader(str(path), encoding="utf-8").load()
        for doc in docs:
            doc.metadata["source"] = path.name
        loaded.extend(docs)

    pdf_path = DOCUMENTS_DIR / "guide_escalade_p1.pdf"
    pdf_docs = PyPDFLoader(str(pdf_path)).load()
    for doc in pdf_docs:
        doc.metadata["source"] = pdf_path.name
    loaded.extend(pdf_docs)

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=450,
        chunk_overlap=80,
        separators=["\n## ", "\n\n", "\n", " ", ""],
    )
    return splitter.split_documents(loaded)


database_dir = OUTPUTS_DIR / "chroma_helpdesk"
vectorstore = Chroma(
    collection_name="helpdesk_documents",
    embedding_function=embeddings,
    persist_directory=str(database_dir),
)

if len(vectorstore.get()["ids"]) == 0:
    helpdesk_chunks = load_helpdesk_chunks()
    vectorstore.add_documents(
        helpdesk_chunks,
        ids=[stable_chunk_id(chunk) for chunk in helpdesk_chunks],
    )

print("Chunks disponibles :", len(vectorstore.get()["ids"]))


## Préparer deux tools indépendants

Le premier applique une règle métier déterministe. Le second consulte la documentation. Chaque outil peut être testé sans agent.


In [ ]:
from langchain_core.tools import tool


@tool
def calculer_sla(priorite: str, utilisateurs_impactes: int) -> str:
    """Calcule le SLA helpdesk à partir de la priorité et du nombre d'utilisateurs impactés."""
    priorite = priorite.strip().upper()
    if utilisateurs_impactes < 0:
        return "Nombre d'utilisateurs invalide."
    if priorite == "CRITIQUE" or utilisateurs_impactes >= 250:
        return "SLA : 1 heure · escalade DSI requise"
    if priorite == "HAUTE":
        return "SLA : 4 heures"
    if priorite == "NORMALE":
        return "SLA : 1 jour ouvré"
    return "Priorité inconnue : validation humaine requise"


@tool
def chercher_procedure(question: str) -> str:
    """Recherche une réponse dans les procédures helpdesk internes et retourne les passages avec leurs sources."""
    docs = vectorstore.similarity_search(question, k=3)
    return "\n\n".join(
        f"[Source : {doc.metadata.get('source')}]\n{doc.page_content}"
        for doc in docs
    )


print(calculer_sla.invoke({"priorite": "HAUTE", "utilisateurs_impactes": 20}))
print(chercher_procedure.invoke({"question": "Quel portail VPN saisir ?"})[:300])


## Créer l'agent

`create_agent` construit la boucle autour du modèle et des tools. Le `system_prompt` précise les règles communes : utiliser les outils, ne pas inventer et rester concis.


In [ ]:
from langchain.agents import create_agent


agent = create_agent(
    model=llm,
    tools=[calculer_sla, chercher_procedure],
    system_prompt=(
        "Tu es un assistant helpdesk. Utilise les tools pour toute procédure ou règle SLA. "
        "N'invente ni procédure, ni source, ni délai. Réponds en français et reste concis."
    ),
)


## Observer le cycle sans demander la pensée privée

Nous inspectons les événements observables : messages, demandes d'outil, résultats d'outil et réponse finale. Nous n'avons pas besoin d'exposer un raisonnement interne détaillé.


![Boucle décision, action et observation](../assets/jour_01/11_cycle_agent.png)

*Le résultat du tool revient au modèle, qui peut répondre ou choisir une autre action.*


In [ ]:
resultat_sla = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Quel est le SLA d'un ticket CRITIQUE qui bloque 400 personnes ?",
    }]
})

for message in resultat_sla["messages"]:
    appels = getattr(message, "tool_calls", [])
    if appels:
        print("APPEL TOOL :", [(appel["name"], appel["args"]) for appel in appels])
    if message.type == "tool":
        print("OBSERVATION :", message.content)

print("RÉPONSE :", resultat_sla["messages"][-1].content)


## Tester un autre chemin

Cette fois, la question demande une procédure. L'agent devrait choisir `chercher_procedure`, pas `calculer_sla`.


In [ ]:
resultat_rag = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Quel portail faut-il saisir dans GlobalProtect ?",
    }]
})

for message in resultat_rag["messages"]:
    appels = getattr(message, "tool_calls", [])
    if appels:
        print("APPEL TOOL :", [appel["name"] for appel in appels])

print("RÉPONSE :", resultat_rag["messages"][-1].content)


### À vous de jouer — définir les attentes

Préparez des cas où l'outil attendu est connu à l'avance. Une salutation ne devrait nécessiter aucun tool.


In [ ]:
# À vous de jouer — ajoutez trois questions et l'outil attendu, ou None.
cas_agent = []
print("Cas préparés :", len(cas_agent))


## Ajouter une mémoire courte avec un checkpointer

Le **checkpointer** conserve l'état d'une conversation. Le même `thread_id` reprend le même fil ; un autre identifiant crée une conversation séparée.


In [ ]:
from langgraph.checkpoint.memory import InMemorySaver


agent_avec_memoire = create_agent(
    model=llm,
    tools=[calculer_sla, chercher_procedure],
    system_prompt=(
        "Tu es un assistant helpdesk. Utilise les tools pour les faits métier. "
        "N'invente rien et réponds en français."
    ),
    checkpointer=InMemorySaver(),
)

config = {
    "configurable": {"thread_id": "demo-vpn-001"},
    "recursion_limit": 8,
}

premier_tour = agent_avec_memoire.invoke(
    {"messages": [{"role": "user", "content": "Mon problème concerne GlobalProtect."}]},
    config=config,
)
second_tour = agent_avec_memoire.invoke(
    {"messages": [{"role": "user", "content": "Quel portail dois-je saisir ?"}]},
    config=config,
)

print(second_tour["messages"][-1].content)


## Limites et garde-fous

- Limiter le nombre d'itérations avec `recursion_limit`.
- Garder les tools étroits et validés.
- Tester le choix d'outil séparément de la qualité de la réponse.
- Exiger une confirmation humaine avant une action externe importante.
- Journaliser les appels de tools sans enregistrer de secrets.

## À retenir

Un agent est utile lorsqu'un choix dynamique est nécessaire. Sinon, une chaîne déterministe reste plus simple à comprendre, tester et sécuriser.
